# 1) Spin-the-Wheel Study Task Picker

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Help a student start quickly by randomly picking one 10-minute study task and showing a simple challenge.

## Simple rules (policy)

- Pick one task per loop.
- Give a 10-minute challenge.
- Track pick counts in memory (JSON).

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

    @staticmethod
    def save_csv(path: str, picks: Dict[str, int]) -> str:
        try:
            import csv
            with open(path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['task', 'count'])
                for task, count in sorted(picks.items(), key=lambda kv: -kv[1]):
                    writer.writerow([task, count])
            return 'csv_saved'
        except Exception as e:
            print('Error saving CSV:', e)
            return 'csv_error'

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "export_csv":
            picks = memory.get('picks', {})
            csv_path = os.path.splitext(self.memory_path)[0] + '.csv'
            res = Tools.save_csv(csv_path, picks)
            return {"result": ("exported_csv" if res == 'csv_saved' else 'csv_error'), "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "01_spin_wheel_memory.json"

# Tasks now include difficulty tags (easy/medium/hard)
TASKS_DEFAULT = [
    {"task": "Revise last lecture notes", "difficulty": "medium"},
    {"task": "Solve 2 easy problems", "difficulty": "easy"},
    {"task": "Rewrite one concept in your own words", "difficulty": "medium"},
    {"task": "Make 5 flashcards", "difficulty": "easy"},
    {"task": "Refactor one function name + variables", "difficulty": "hard"},
]

# Weights used for weighted random selection (by difficulty)
WEIGHT_BY_DIFFICULTY = {"easy": 1.0, "medium": 1.2, "hard": 1.5}

def _format_task(t: Dict[str, Any]) -> str:
    return f"{t['task']} ({t.get('difficulty', 'medium')})"

def decide_next_action(user_text: str, memory: Dict[str, Any]) -> Action:
    if should_stop(user_text):
        return Action("terminate", {})

    # ensure tasks stored as list of dicts
    tasks = memory.get("tasks") or [t.copy() for t in TASKS_DEFAULT]
    memory["tasks"] = tasks
    picks = memory.get("picks", {})

    cmd = (user_text or "").strip()
    cmd_lower = cmd.lower()

    # Commands: empty -> spin; add:..., remove:..., list, leaderboard, export
    if cmd == "":
        # Spin the wheel with weighted choices and avoid immediate repeats
        names = [t['task'] for t in tasks]
        diffs = [t.get('difficulty', 'medium') for t in tasks]
        weights = [WEIGHT_BY_DIFFICULTY.get(d, 1.0) for d in diffs]
        last = memory.get("last_choice")
        if last in names and len(names) > 1:
            weights = [0 if n == last else w for n, w in zip(names, weights)]
            if all(w == 0 for w in weights):
                weights = [1.0] * len(names)
        choice = random.choices(names, weights=weights, k=1)[0]
        picks[choice] = int(picks.get(choice, 0)) + 1
        memory["picks"] = picks
        memory["last_choice"] = choice
        msg = (
            f"Your task wheel picked: {choice}\n"
            "Challenge: do it for 10 minutes (no switching).\n"
            f"Times picked: {picks[choice]}"
        )
        return Action("notify", {"message": msg})

    # Add a task inline: 'add: Task description | difficulty' (difficulty optional)
    if cmd_lower.startswith("add:"):
        payload = cmd.split(":", 1)[1].strip()
        parts = [p.strip() for p in payload.split("|", 1)]
        task_text = parts[0] if parts else ""
        difficulty = parts[1].lower() if len(parts) > 1 else "medium"
        if difficulty not in WEIGHT_BY_DIFFICULTY:
            difficulty = "medium"
        tasks.append({"task": task_text, "difficulty": difficulty})
        memory["tasks"] = tasks
        return Action("notify", {"message": f"Added task: {task_text} ({difficulty})"})

    # Remove a task inline: 'remove: NAME' or 'remove: INDEX' (1-based)
    if cmd_lower.startswith("remove:"):
        payload = cmd.split(":", 1)[1].strip()
        # try index
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(tasks):
                removed = tasks.pop(idx)
                memory["tasks"] = tasks
                return Action("notify", {"message": f"Removed: {_format_task(removed)}"})
            return Action("notify", {"message": "Index out of range."})
        # try exact name
        for i, t in enumerate(tasks):
            if t['task'].lower() == payload.lower():
                removed = tasks.pop(i)
                memory["tasks"] = tasks
                return Action("notify", {"message": f"Removed: {_format_task(removed)}"})
        return Action("notify", {"message": "Task not found."})

    if cmd_lower in {"list", "tasks"}:
        lines = [f"{i+1}. {_format_task(t)}" for i, t in enumerate(tasks)]
        return Action("notify", {"message": "Tasks:\n" + "\n".join(lines)})

    if cmd_lower in {"leaderboard", "top"}:
        if not picks:
            return Action("notify", {"message": "No picks yet."})
        sorted_p = sorted(picks.items(), key=lambda kv: -kv[1])
        lines = [f"{task}: {count}" for task, count in sorted_p]
        return Action("notify", {"message": "Leaderboard:\n" + "\n".join(lines)})

    if cmd_lower.startswith("export"):
        # returns an action the environment knows how to handle
        return Action("export_csv", {})

    # Unknown command -> show help
    help_msg = (
        "Commands:\n"
        "  (Enter) - spin the wheel\n"
        "  add: TASK | difficulty - add a task (difficulty optional)\n"
        "  remove: INDEX|NAME - remove by 1-based index or exact name\n"
        "  list or tasks - show tasks\n"
        "  leaderboard - show pick counts\n"
        "  export - save picks to CSV\n"
        "  stop - exit and save stats\n"
    )
    return Action("notify", {"message": "Unknown command.\n" + help_msg})

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}

    Tools.notify("Spin-the-Wheel Task Picker is running.")
    Tools.notify("Press Enter to spin. Type 'stop' to end. Type 'list' to view tasks.")

    user_text = Tools.ask("Ready? (Enter to spin)")
    while True:
        action = decide_next_action(user_text, memory)
        out = env.execute(action, memory)
        memory = out["memory"]

        if out["result"] == "terminated":
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved stats. Bye!")
            break

        # handle export action result (Environment writes the CSV)
        if out["result"] == "exported_csv":
            Tools.notify("Exported picks to CSV.")

        env.execute(Action("save_memory", {}), memory)
        user_text = Tools.ask("Spin again? (Enter / stop / commands)")

run_agent()


## Easy extensions

- Let students add/remove tasks during runtime (use `add: TASK | difficulty` and `remove: INDEX|NAME`).
- Add difficulty tags (easy/medium/hard) to tasks and use weighted picks.
- Print a small leaderboard of most-picked tasks via `leaderboard`.
- Prevent immediate repeats (won't pick same task twice in a row).
- Export pick stats to CSV: use `export` (creates `01_spin_wheel_memory.csv`).
- List tasks with indices using `list` or `tasks`.

Usage examples:

- `add: Read summary | easy`  -> adds a new easy task.
- `remove: 2`               -> removes task at index 2.
- `leaderboard`              -> shows most-picked tasks.